# Evaluation Demo - Testing Basic Questions

This notebook demonstrates how to use the evaluation set and verify retrieval performance.

## Load Evaluation Set

In [7]:
import json
from pathlib import Path
import pandas as pd

# Load evaluation questions
eval_path = Path("../data/evaluation/basic_questions.jsonl")
questions = []

with open(eval_path, "r") as f:
    for line in f:
        questions.append(json.loads(line))

print(f"Loaded {len(questions)} evaluation questions")

# Convert to DataFrame for easy viewing
df = pd.DataFrame(questions)
print(f"\nEvaluation set overview:")
print(df[['question_id', 'company', 'risk_profile', 'requires_exact_match', 'difficulty']].head(10))

Loaded 15 evaluation questions

Evaluation set overview:
  question_id          company risk_profile  requires_exact_match difficulty
0        b001         barclays  adventurous                 False       easy
1        b002         barclays     balanced                  True       easy
2        b003         barclays       growth                  True       easy
3        b004  scottish_widows          n/a                 False       easy
4        b005  scottish_widows          n/a                  True       easy
5        b006  scottish_widows  adventurous                  True       easy
6        b007           lloyds     balanced                 False       easy
7        b008           lloyds     cautious                  True       easy
8        b009           lloyds  adventurous                 False       easy
9        b010          natwest     cautious                  True       easy


## Connect to Retrieval System

In [8]:
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))

from retrieval.embedding import EmbeddingGenerator
from retrieval.qa_index import QdrantIndexer

# Initialize
print("Loading embedding model...")
embedder = EmbeddingGenerator(model_name="all-MiniLM-L6-v2")

print("Connecting to Qdrant...")
indexer = QdrantIndexer(
    collection_name="uk_investment_rag",
    persistence_path=Path("../data/qdrant_storage"),
    in_memory=False
)

print("✓ Ready for evaluation!")

/Users/fuli/Desktop/RAG_project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-26 00:18:39,710 - INFO - Loading embedding model: all-MiniLM-L6-v2
2026-04-26 00:18:39,847 - INFO - Use pytorch device_name: mps
2026-04-26 00:18:39,848 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loading embedding model...


2026-04-26 00:18:42,333 - INFO - Embedding dimension: 384
2026-04-26 00:18:42,341 - INFO - Initializing persistent Qdrant client at ../data/qdrant_storage


Connecting to Qdrant...


RuntimeError: Storage folder ../data/qdrant_storage is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

## Run Retrieval on Evaluation Set

In [9]:
def evaluate_question(question_data, limit=3):
    """Evaluate a single question and return results."""
    question = question_data['question']
    question_id = question_data['question_id']
    
    # Generate embedding
    query_embedding = embedder.generate_embeddings([question])[0]
    
    # Search
    results = indexer.search(query_embedding, limit=limit)
    
    return {
        "question_id": question_id,
        "question": question,
        "company": question_data.get('company'),
        "requires_exact_match": question_data.get('requires_exact_match'),
        "relevance_type": question_data.get('relevance_type'),
        "top_results": results,
        "top_score": results[0]['score'] if results else 0,
        "top_company": results[0]['payload']['company'] if results else None
    }

# Evaluate all questions
evaluation_results = []

for q_data in questions:
    result = evaluate_question(q_data)
    evaluation_results.append(result)
    print(f"{result['question_id']}: {result['top_score']:.4f} ({result['top_company']})")

print(f"\n✓ Evaluated {len(evaluation_results)} questions")

2026-04-26 00:18:47,437 - INFO - Generating embeddings for 1 texts...
Batches: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it]


NameError: name 'indexer' is not defined

## Analyze Results

In [ ]:
# Convert to DataFrame for analysis
results_df = pd.DataFrame(evaluation_results)

# Overall statistics
print("=== Overall Statistics ===")
print(f"Average top score: {results_df['top_score'].mean():.4f}")
print(f"Median top score: {results_df['top_score'].median():.4f}")
print(f"Min top score: {results_df['top_score'].min():.4f}")
print(f"Max top score: {results_df['top_score'].max():.4f}")

# By company
print("\n=== By Company ===")
company_stats = results_df.groupby('company')['top_score'].agg(['mean', 'count'])
print(company_stats)

# By relevance type
print("\n=== By Relevance Type ===")
relevance_stats = results_df.groupby('relevance_type')['top_score'].agg(['mean', 'count'])
print(relevance_stats)

# Exact match vs non-exact match
print("\n=== Exact Match Required ===")
exact_match_stats = results_df.groupby('requires_exact_match')['top_score'].agg(['mean', 'count'])
print(exact_match_stats)

## Detailed Results Inspection

In [ ]:
def show_detailed_results(result):
    """Show detailed results for a question."""
    print(f"\n{'='*80}")
    print(f"Question ID: {result['question_id']}")
    print(f"Question: {result['question']}")
    print(f"Company: {result['company']}")
    print(f"Relevance Type: {result['relevance_type']}")
    print(f"Requires Exact Match: {result['requires_exact_match']}")
    print(f"\nTop {len(result['top_results'])} Results:")
    
    for i, r in enumerate(result['top_results'], 1):
        print(f"\n  Result {i} (score: {r['score']:.4f})")
        print(f"    Company: {r['payload']['company']}")
        print(f"    Product: {r['payload']['product_type']}")
        print(f"    Risk Profile: {r['payload']['risk_profile']}")
        print(f"    Content: {r['payload']['content'][:150]}...")

# Show detailed results for a few questions
for i in [0, 5, 10, 14]:  # Show examples from different companies
    show_detailed_results(evaluation_results[i])

## Save Results

In [ ]:
# Save evaluation results
results_path = Path("../data/evaluation_results/basic_dense_baseline.json")
results_path.parent.mkdir(parents=True, exist_ok=True)

with open(results_path, "w") as f:
    json.dump(evaluation_results, f, indent=2)

print(f"✓ Saved results to {results_path}")

## Human Verification Helper

This section helps you manually verify if the retrieved chunks contain the expected answers.

In [ ]:
def verify_answer(question_data, top_result):
    """Helper function to manually verify if answer is in retrieved content."""
    print(f"\n{'='*80}")
    print(f"Question: {question_data['question']}")
    print(f"\nExpected Answer:")
    print(f"  {question_data['expected_answer']}")
    print(f"\nTop Retrieved Content (score: {top_result['score']:.4f}):")
    print(f"  {top_result['payload']['content'][:500]}...")
    print(f"\n❓ Does the retrieved content contain the expected answer? (Y/N)")
    
    # You can manually verify and note down results
    return top_result

# Verify a few key questions
key_questions = ['b002', 'b005', 'b008', 'b012', 'b015']  # Questions requiring exact match

for q_id in key_questions:
    q_data = next(q for q in questions if q['question_id'] == q_id)
    result = next(r for r in evaluation_results if r['question_id'] == q_id)
    if result['top_results']:
        verify_answer(q_data, result['top_results'][0])